# 1. Prepare the Workstation

In [2]:
# Import time to track runtime 
import time
notebook_start = time.time()
prepare_start = time.time()
library_start = time.time()

## Libraries

In [3]:
# Library Imports

## Import Required Libraries
import os
from dotenv import load_dotenv
import sys
os.environ["OMP_NUM_THREADS"] = '4'
import requests
from io import StringIO

# Loops
from itertools import product

# Python output libraries
from IPython.display import clear_output
from tqdm import tqdm
# Permit tqdm bars on pandas apply method
tqdm.pandas()

# Core DataFrame/Series
import pandas as pd
from pandas.api.types import is_numeric_dtype

# Geo Libraries
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
import pyproj
from pyproj import CRS


# Mathematical libraries
import numpy as np
from scipy import stats
from scipy.stats import kurtosis
from scipy.stats import median_abs_deviation

# Time libraries
from dateutil import parser
from datetime import datetime
from datetime import date, timedelta, datetime

# Webscraping libraries
import requests
import glob

# Visualisation libraries
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import matplotlib.patches as mpatches

# For Exporting / Importing Models
import pickle

# Other text analysis tools
from collections import Counter

# For VIF feature testing
from patsy import dmatrices
from statsmodels.stats.outliers_influence import variance_inflation_factor

# For RFE feature testing
import sklearn
from sklearn.datasets import make_regression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

# Automating PowerPoints
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE
from pptx.util import Inches, Pt

# Verify versions
print(f'Numpy Version: {np.__version__}')
print(f'Pandas Version: {pd.__version__}')
print(f'GeoPandas:', gpd.__version__)
print(f'Matplotlib Version: {matplotlib.__version__}')
print(f'Seaborn Version: {sns.__version__}')
print(f'sklearn Version: {sklearn.__version__}')

# Library Section Runtime
library_runtime = time.time() - library_start
print(f'\nTotal Library Import Time:\n{round(library_runtime, 3)}s')

Numpy Version: 2.3.5
Pandas Version: 3.0.0
GeoPandas: 1.1.2
Matplotlib Version: 3.10.8
Seaborn Version: 0.13.2
sklearn Version: 1.8.0

Total Library Import Time:
4.283s


## General Functions

In [4]:
# Tracking runtime
function_start = time.time()

In [5]:
## Core Exploration and validation Functions
# Set dataset specific defaults
# (see exploration section)
common_value = ''

# Core DataFrame statistics
def df_explore(df, prnt=True):
    '''
    Output
        Return statistics as DataFrame
        for greater readability:
    Optional: 
        print shape, type count, 
        & column name list.
        Output first 3 rows
    '''
    # Calculate 
    df_types = pd.DataFrame(df.dtypes)
    df_nulls = pd.DataFrame(df.isna().sum())

    # Horizontal concatenate (no sorting required)
    df_explored = pd.concat([df_types, df_nulls], axis=1)
 
    # Reassign column names:
    col_names = ['Data_Type', 'Null_Count']
    df_explored.columns = col_names
    df_explored.index.name = 'Column'


    # Add additional statistics
    uniq_list = []
    sum_list = []
    min_list = []
    max_list = []
    mean_list = []
    median_list = []
    std_list=[]
    skew_list = []
    
    for col in df:
        uniq_list.append(df[col].nunique())
        if is_numeric_dtype(df[col]):
            sum_list.append(round(df[col].sum(), 3))
            min_list.append(df[col].min())
            max_list.append(df[col].max())
            mean_list.append(round(df[col].mean(),3))
            median_list.append(round(df[col].median(),3))
            std_list.append(round(df[col].std(),3))
            skew_list.append(round(df[col].skew(), 3))
        elif df[col].dtypes == '<M8[ns]' \
        or df[col].dtypes == 'period[M]' \
        or df[col].dtypes == 'datetime64[s]':
            min_list.append(df[col].min())
            max_list.append(df[col].max())
            sum_list.append('N/A')
            mean_list.append('N/A')
            median_list.append('N/A')
            std_list.append('N/A')
            skew_list.append('N/A')
        else:
            sum_list.append('N/A')
            min_list.append('N/A')
            max_list.append('N/A')
            mean_list.append('N/A')
            median_list.append('N/A')
            std_list.append('N/A')
            skew_list.append('N/A')

    stats = pd.concat([
        pd.Series(uniq_list).rename('Unique'),
        pd.Series(sum_list).rename('Sum'),        
        pd.Series(min_list).rename('Min'),
        pd.Series(max_list).rename('Max'),
        pd.Series(mean_list).rename('Mean'),
        pd.Series(median_list).rename('Median'),
        pd.Series(std_list).rename('Standard Deviation'),
        pd.Series(skew_list).rename('Skew'),     
    ], axis=1) 
    
    stats.index = df.columns
    
    df_explored = df_explored \
                .merge(
                    stats, how='left', 
                    left_index=True, right_index=True
                ).reset_index()
    
    if prnt:
        # Print key Information 
        print(f'SHAPE:\nCOLUMNS: {df.shape[1]} \
              \nROWS: {df.shape[0]} \
              \nDUPLICATE ROWS: {df.duplicated().sum()}\n \
              \nTYPE COUNT:\n{df.dtypes.value_counts()}\n \
              \nCOLUMN LIST:\n{list(df)}\n\n'
             )
        # Display statistics and example rows
        display(df_explored)
        display(df.head(3))
        return
    else:
        return df_explored

# Display information on unique values in columns
def df_count_uniq(df, col=False, prnt=False):
    '''
    Intended for categorical fields
    Input
        The column name must also be specified (col)
        Optional to separately set as col  
    Output
        List unique values, with count and proportion
        Returned as DataFrame for greater readability
    Optional:
        Print number of unique values as reference
    CANSCRAPAdded check for Pandas version issue - older version 
    has a different output for value_counts
    '''
    if col:
        df = df[col]
    
    if prnt:
        print(f'Unique values ({df.nunique()} / {df.shape[0]}): \
                \n{pd.unique(df)}'
             )

    # Create the output DataFrame
    uniques = pd.DataFrame(df.value_counts())
    
    '''
    # CANSCRAP If encountering the pandas version issue
    if uniques.shape[1] == 1:
        col = list(uniques)[0]
        uniques = uniques.reset_index(drop=False)
        uniques.columns = [col, 'count']
        uniques['proportion %'] = (
            100*df.value_counts(normalize=True).round(4).values)
        return uniques
    '''
    
    uniques['proportion %'] = 100*pd.DataFrame(
        df.value_counts(normalize=True).round(4)
    )
    return uniques.reset_index()

common_value = ''


# Needs tested for older pandas version
def df_sum_uniq(df, group_col, value_col=common_value,
                  prnt=False, ascend=False):
    '''
    Intended for categorical fields
    Input
        The column name must also be specified (group_col) 
    Output
        Returned as DataFrame for greater readability
    Optional:
        Print number of unique values as reference
    '''
    if prnt:
        print(f'Unique values ({df[group_col].nunique()} / {df.shape[0]}): \
                \n{pd.unique(df[group_col])}'
             )

    # Create the output DataFrame
    uniques = df_group_sum_sort(df, group_col, sort_col=value_col,
                             value_col=value_col, ascend=ascend)
    uniques['Proportion (%)'] = (
        (100 * uniques[value_col] / uniques[value_col].sum()).round(2)
        )
    uniques = uniques.rename(columns={value_col:'total_sum_'+value_col})
    
    return uniques.reset_index(drop=True)

## Data Cleaning
# Convert remaining objects to string type and clean
def clean_string(s):
    """
    Strip leading/trailing whitespace.
    Remove any double spaces.
    Returns None if input is NaN or None.
    """ 
    if s is None:
        return None
    # Strip whitespace
    return str(s).strip().replace('  ', ' ')

def df_clean_col(df, col=False):
    '''
    Output:
        Remove leading/trailing
        whitespace from specified col
        and remove any double spaces.
    '''    
    if is_numeric_dtype(df[col]):
        print(f'{col}: Numeric Type')
        return    
    else:
        df[col] = df[col].map(clean_string)
        return df

def df_recast_obj_to_str(df, clean=True):
    '''
    Output:
        Trim any leading/trailing
        whitespace and set as Proper case.
        Re-cast all remaining object-type
        columns as strings.
        
        For use after all specific datatype
        changes have been made.
    '''
    for col in df:
        if df[col].dtype == 'object':
            if clean:
                df = df_clean_col(df, col)            
            df[col] = df[col].astype('string')
    return df

## Outlier Analysis
# IQR, MAD & Standard Deviation Outlier functions
def df_calc_iqr_limits(df, 
                value_col=common_value,
                coefficient=1.5, prnt=False):
    '''
    Output:
        Upper & lower IQR outlier limits
        Default 1.5 IQR
    Optional:
        Print number and percentage of outliers
    '''
    # Using np quantile as default pandas uses inclusive method
    q1 = np.quantile(df[value_col], method='weibull', q=.25)
    q3 = np.quantile(df[value_col], method='weibull', q=.75) 

    iqr = q3 - q1

    upper = q3 + coefficient*iqr
    lower = q1 - coefficient*iqr

    # Optional print outlier count
    if prnt:
        total = df[value_col].count()
        above_cnt = df[df[value_col]>upper].shape[0]
        below_cnt = df[df[value_col]<lower].shape[0]
        above_percent = (100*above_cnt/total).round(2)
        below_percent = (100*below_cnt/total).round(2)
        
        print(f'>{coefficient}*IQR Outliers Count: \
                \nAbove Upper Limit ({round(upper, 3)}): \
                \n{above_cnt} ({above_percent}%) \
                \nBelow Lower Limit ({round(lower, 3)}): \
                \n{below_cnt} ({below_percent}%) \
                ')

    return upper, lower


def df_calc_zsd_limits(df,
                value_col=common_value,
                coefficient=3, prnt=False):
    '''
    Output:
        Upper & lower outlier limits
        Default 3x Standard Deviations
    Optional:
        Print number and percentage of outliers
    '''
    mean = df[value_col].mean()
    std = df[value_col].std()
    
    upper = mean + coefficient * std
    lower = mean - coefficient * std

    # Optional print outlier count
    if prnt:
        total = df[value_col].count()
        above_cnt = df[df[value_col]>upper].shape[0]
        below_cnt = df[df[value_col]<lower].shape[0]
        above_percent = (100*above_cnt/total).round(2)
        below_percent = (100*below_cnt/total).round(2)
        
        print(f'>{coefficient}*STD Outliers Count: \
                \nAbove Upper Limit ({round(upper, 3)}): \
                \n{above_cnt} ({above_percent}%) \
                \nBelow Lower Limit ({round(lower, 3)}): \
                \n{below_cnt} ({below_percent}%) \
                ')
    
    return upper, lower


def df_calc_mad_limits(df,
                value_col=common_value,
                threshold=3.5,
                prnt=False):
    '''
    Source:
        https://www.itl.nist.gov/div898/handbook/eda/section3/eda35h.htm
        Reverse the equation to get the values at the threshold 
    Output:
        Upper & lower outlier limits using MAD method.
        Default 3.5 threshold
    Optional:
        Print number and percentage of outliers
        above the threshold
    '''
    median = df[value_col].median()
    mad = median_abs_deviation(df[value_col])

    mad_term = (mad * threshold) / 0.6745
    
    upper = median + mad_term
    lower = median - mad_term

    # Optional print outlier count
    if prnt:
        total = df[value_col].count()
        above_cnt = df[df[value_col]>upper].shape[0]
        below_cnt = df[df[value_col]<lower].shape[0]
        above_percent = (100*above_cnt/total).round(2)
        below_percent = (100*below_cnt/total).round(2)
        
        print(f'>3.5 MAD Outliers Count: \
                \nAbove Upper Limit ({round(upper, 3)}): \
                \n{above_cnt} ({above_percent}%) \
                \nBelow Lower Limit ({round(lower, 3)}): \
                \n{below_cnt} ({below_percent}%) \
                ')
    
    return upper, lower


def df_extract_outliers(df,
                        value_col=common_value,
                        method='zsd', outliers_only=False,
                        **kwargs):
    '''
    Output:
        Identify and remove the outliers in the
        specified DataFrame.
        Default to the 3*sd method.
    Optional:
        Return only the outliers for analysis.
    '''
    if method == 'zsd':
        upper, lower = df_calc_zsd_limits(df,
                                          value_col=value_col,
                                          **kwargs)
    elif method == 'iqr':
        upper, lower = df_calc_iqr_limits(df,
                                          value_col=value_col,
                                          **kwargs)
    elif method == 'mad':
        upper, lower = df_calc_mad_limits(df,
                                          value_col=value_col,
                                          **kwargs)
        
    df_filter = (
        (df[value_col] > upper) | (df[value_col] < lower)
    )
    
    if outliers_only:
        return df.loc[df_filter]
    else:
        return df.loc[~df_filter]
    

In [6]:
## Visualisation
# Style Functions
def reset_visual_styles():
    '''Re-apply preffered styles'''
    sns.set_style('darkgrid')
    sns.set_palette('colorblind')

reset_visual_styles()

def clean_label(label):
    ''' 
    Basic cleaning on inputted label.
    '''
    return label.replace('_', ' ').title()

# Plot histogram, box and violin charts of selected
def plot_basic_distribution(df, col, prnt=None,
                            title='Distribution Plots',
                            stat='probability',
                            showmeans=True,
                            **kwargs):
    '''
    Plot histogram, box and violin charts of selected
    Dataframe[columm] to show basic distribution.
    Optional print of Skew, Max & Min information.
    '''
    # Print basic information
    # Only print if required
    if prnt:
        print(f'Column: {col}\
                \nMean: {round(df[col].mean(),3)}\
                \nMedian: {round(df[col].median(),3)} \
                \nSkew: {round(df[col].skew(),3)} \
                \nMax: {round(df[col].max(),3)} \
                \nMin: {round(df[col].min(),3)} \
                \n')
        df_calc_iqr_limits(df, col, prnt=prnt)

    # Set figure with 3 rows
    fig, ax = plt.subplots(3)
    fig.set_size_inches(8, 6)
    
    sns.histplot(data=df, x=col, stat=stat,
                 **kwargs, ax=ax[0])
    sns.boxplot(data=df, x=col, ax=ax[1],
                showmeans=showmeans,
                meanprops={'marker': '+',
                          'markeredgecolor': 'black',
                          'markersize': '10',
                          'label': 'mean'})
    sns.violinplot(data=df, x=col, ax=ax[2])

    ax[0].set(xlabel='', title=title)
    if showmeans:
        ax[0].axvline(x=df[col].mean(),
                      color='black',
                      label='mean')
        ax[0].legend()
    ax[1].set(xlabel='')
    ax[2].set(xlabel=col)
    
    fig.tight_layout()

def group_distributions(df, category_col,
                        value_col=common_value,
                        showmeans=True,
                        **kwargs):
    '''
    Plot box/violin plot distributions split by group
    '''
    
    # Set figure size with 2 rows
    fig, ax = plt.subplots(2)
    fig.set_size_inches(12, 6)
    
    # Plot box & violin
    sns.boxplot(data=df, y=value_col, hue=category_col,
                notch=True, dodge=True, ax=ax[0],
                showmeans=showmeans, **kwargs)
    sns.violinplot(data=df, y=value_col, hue=category_col,
                   dodge=True, ax=ax[1])

    
    ax[0].set(title=(
        'Distributions of ' +value_col+ ' by ' +category_col))
    ax[0].legend(bbox_to_anchor=(1, .75))
    ax[1].legend(bbox_to_anchor=(1, .75))
    
    fig.tight_layout()

In [7]:
## Other Supporting Functions
# Deduplicate a list
def dedup_list(xlist):
    return list(dict.fromkeys(xlist))

# Create date range for bulk queries (inclusive)
def generate_str_date_range(start_date, end_date,
                        date_format = '%Y-%m-%d',
                        date_range = False):
    '''
    Inputs:
        Date must be string (YYYY-MM-DD format)
    Actions:
        Generate string list between inputted dates
        in inputted format.
    Optional:
        Input existing date_range list to append to.
    '''   
    # If date inputs are string, convert to datetime
    if isinstance(start_date, str):
        try:
            start_date =  parser.parse(start_date)
        except:    
            return print('Formating Error: start_date')
            
    if isinstance(end_date, str):
        try:
            end_date =  parser.parse(end_date)
        except:    
            return print('Formating Error: end_date')        
    if start_date > end_date:
        return print('Error: End Date before Start Date ')
    
    # Calculate length of required strings
    delta = end_date - start_date

    # If no existing list is provided,
    # create a blank list.
    if not date_range:
        date_range = list()

    # Loop through time period by day and append to list.
    for i in range(delta.days + 1):
        day = start_date + timedelta(days=i)
        date_range.append(day.strftime(date_format))

    return date_range

# Test Function
start_date = '2024-09-01'
end_date = '2024-11-30'

generate_str_date_range(start_date, end_date)

# Perform basic VIF testing for feature selection
def vif_test(df, dep_col, indep_cols):

    #find design matrix for linear regression model using 'rating' as response variable 
    y, X = dmatrices(
        f'{dep_col} ~ {'+'.join(indep_cols)}',
        data=df, return_type='dataframe')
    
    #calculate VIF for each explanatory variable
    vif_df = pd.DataFrame()
    vif_df['VIF'] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])]
    vif_df['variable'] = X.columns
    
    #view VIF for each explanatory variable 
    return vif_df.sort_values(by='VIF').round(3).reset_index(drop=True)

# Create folder if not already present
def create_folder(folder_path):

    if os.path.exists(folder_path):
        print(f'Folder already exists: {folder_path}')
        return
    else:
        os.makedirs(folder_path)
        print(f'Folder created: {folder_path}')
        return

# Function test
#create_folder("Experiments/test_folder/")

# Grouping functions
def df_group_sum_sort(df, group_col, sort_col=False,
                      value_col=common_value,
                      ascend=True):
    '''
    Group, sum and sort the inputted data frame
    '''
    if isinstance(group_col, str):
        group_col = [group_col]
            
    if not sort_col:
        sort_col = group_col
    
    return df[group_col + [value_col]] \
            .groupby(group_col, observed=True) \
            .sum() \
            .sort_values(by=sort_col, ascending=ascend) \
            .reset_index()

## Runtime

In [8]:
# Print Runtime
function_runtime = time.time() - function_start
print(f'General Functions:\n{round(function_runtime, 3)}s')

General Functions:
0.091s


# Runtime

In [19]:
## Print Runtime
function_runtime = time.time() - function_start
print(f'Library Import Time:\n{round(library_runtime, 3)}s')

print(f'Function Time:\n{round(function_runtime, 3)}s')

prepare_runtime = round(time.time() - prepare_start, 3)
print(f'\nStage Total Runtime:\n{prepare_runtime}s')

Library Import Time:
4.283s
Function Time:
5.249s

Stage Total Runtime:
9.542s
